In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("Telco_customer_churn_dataset.csv")
df.head()

,Age,Avg Monthly GB Download,Avg Monthly Long Distance Charges,Churn Category,Churn Reason,Churn Score,City,CLTV,Contract,Country,...,Tenure in Months,Total Charges,Total Extra Data Charges,Total Long Distance Charges,Total Refunds,Total Revenue,Under 30,Unlimited Data,Zip Code,Churn
0,72,4,19.44,NaN,NaN,51,San Mateo,4849,Two Year,United States,...,25,2191.15,0,486.00,0.0,2677.15,0,1,94403,0
1,27,59,45.62,NaN,NaN,27,Sutter Creek,3715,Month-to-Month,United States,...,35,3418.20,0,1596.70,0.0,5014.90,1,1,95685,0
2,59,0,16.07,NaN,NaN,59,Santa Cruz,5092,Month-to-Month,United States,...,46,851.20,0,739.22,0.0,1590.42,0,0,95064,0
3,25,27,0.00,NaN,NaN,49,Brea,2068,One Year,United States,...,27,1246.40,30,0.00,0.0,1276.40,1,0,92823,0
4,31,21,17.22,Dissatisfaction,Network reliability,88,San Jose,4026,One Year,United States,...,58,3563.80,0,998.76,0.0,4562.56,0,1,95117,1


In [2]:
df = df.drop( columns = ["Churn Category", "Churn Reason" , "Churn Score" , "Lat Long" , "State" , "City" , "Customer Status" , "Customer ID" , "Zip Code", "Country"])

In [3]:
df.columns.tolist()

['Age',
 'Avg Monthly GB Download',
 'Avg Monthly Long Distance Charges',
 'CLTV',
 'Contract',
 'Dependents',
 'Device Protection Plan',
 'Gender',
 'Internet Service',
 'Internet Type',
 'Latitude',
 'Longitude',
 'Married',
 'Monthly Charge',
 'Multiple Lines',
 'Number of Dependents',
 'Number of Referrals',
 'Offer',
 'Online Backup',
 'Online Security',
 'Paperless Billing',
 'Partner',
 'Payment Method',
 'Phone Service',
 'Population',
 'Premium Tech Support',
 'Quarter',
 'Referred a Friend',
 'Satisfaction Score',
 'Senior Citizen',
 'Streaming Movies',
 'Streaming Music',
 'Streaming TV',
 'Tenure in Months',
 'Total Charges',
 'Total Extra Data Charges',
 'Total Long Distance Charges',
 'Total Refunds',
 'Total Revenue',
 'Under 30',
 'Unlimited Data',
 'Churn']

In [4]:
df.dtypes


Age                                    int64
Avg Monthly GB Download                int64
Avg Monthly Long Distance Charges    float64
CLTV                                   int64
Contract                              object
Dependents                             int64
Device Protection Plan                 int64
Gender                                object
Internet Service                       int64
Internet Type                         object
Latitude                             float64
Longitude                            float64
Married                                int64
Monthly Charge                       float64
Multiple Lines                         int64
Number of Dependents                   int64
Number of Referrals                    int64
Offer                                 object
Online Backup                          int64
Online Security                        int64
Paperless Billing                      int64
Partner                                int64
Payment Me

In [5]:

df["Offer"].value_counts()

Offer
Offer B    515
Offer E    475
Offer D    341
Offer A    319
Offer C    251
Name: count, dtype: int64

In [6]:
df["Quarter"].value_counts()

Quarter
Q3    4225
Name: count, dtype: int64

In [7]:
df = df.drop(columns = ["Quarter"])

In [8]:
df = pd.get_dummies(df , columns = ["Contract", "Gender", "Internet Type", "Offer", "Payment Method"])

In [9]:
df.dtypes


Age                                    int64
Avg Monthly GB Download                int64
Avg Monthly Long Distance Charges    float64
CLTV                                   int64
Dependents                             int64
Device Protection Plan                 int64
Internet Service                       int64
Latitude                             float64
Longitude                            float64
Married                                int64
Monthly Charge                       float64
Multiple Lines                         int64
Number of Dependents                   int64
Number of Referrals                    int64
Online Backup                          int64
Online Security                        int64
Paperless Billing                      int64
Partner                                int64
Phone Service                          int64
Population                             int64
Premium Tech Support                   int64
Referred a Friend                      int64
Satisfacti

In [10]:
from sklearn.model_selection import train_test_split

X = df.drop(columns = ["Churn"])
y = df["Churn"]

X_train , X_test , y_train , y_test = train_test_split(X,y , test_size=0.2 , random_state=42 , stratify=y)
            

In [11]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Churn
0    0.734615
1    0.265385
Name: proportion, dtype: float64
Churn
0    0.734911
1    0.265089
Name: proportion, dtype: float64


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(class_weight="balanced", max_iter=1000)
log_model.fit(X_train_scaled, y_train)
log_predictions = log_model.predict(X_test_scaled)

print(classification_report(y_test, log_predictions))
print(confusion_matrix(y_test, log_predictions))

              precision    recall  f1-score   support

           0       0.99      0.94      0.96       621
           1       0.86      0.96      0.91       224

    accuracy                           0.95       845
   macro avg       0.92      0.95      0.94       845
weighted avg       0.95      0.95      0.95       845

[[586  35]
 [  8 216]]


In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

print(classification_report(y_test, rf_predictions))
print(confusion_matrix(y_test, rf_predictions))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98       621
           1       0.99      0.91      0.94       224

    accuracy                           0.97       845
   macro avg       0.98      0.95      0.96       845
weighted avg       0.97      0.97      0.97       845

[[618   3]
 [ 21 203]]


In [14]:
import joblib
joblib.dump(log_model, "churn_model.joblib")


['churn_model.joblib']

In [15]:
df.columns.tolist()

['Age',
 'Avg Monthly GB Download',
 'Avg Monthly Long Distance Charges',
 'CLTV',
 'Dependents',
 'Device Protection Plan',
 'Internet Service',
 'Latitude',
 'Longitude',
 'Married',
 'Monthly Charge',
 'Multiple Lines',
 'Number of Dependents',
 'Number of Referrals',
 'Online Backup',
 'Online Security',
 'Paperless Billing',
 'Partner',
 'Phone Service',
 'Population',
 'Premium Tech Support',
 'Referred a Friend',
 'Satisfaction Score',
 'Senior Citizen',
 'Streaming Movies',
 'Streaming Music',
 'Streaming TV',
 'Tenure in Months',
 'Total Charges',
 'Total Extra Data Charges',
 'Total Long Distance Charges',
 'Total Refunds',
 'Total Revenue',
 'Under 30',
 'Unlimited Data',
 'Churn',
 'Contract_Month-to-Month',
 'Contract_One Year',
 'Contract_Two Year',
 'Gender_Female',
 'Gender_Male',
 'Internet Type_Cable',
 'Internet Type_DSL',
 'Internet Type_Fiber Optic',
 'Offer_Offer A',
 'Offer_Offer B',
 'Offer_Offer C',
 'Offer_Offer D',
 'Offer_Offer E',
 'Payment Method_Bank With

In [16]:
pd.crosstab(df["Internet Service"], df["Internet Type_DSL"] | df["Internet Type_Cable"] | df["Internet Type_Fiber Optic"])

col_0,False,True
Internet Service,,
0,886,0
1,0,3339


In [17]:
# Compute once, from your training dataframe (before scaling)
numeric_defaults = X_train[[
    'Age', 'Avg Monthly GB Download', 'Avg Monthly Long Distance Charges',
    'CLTV', 'Latitude', 'Longitude', 'Number of Dependents', 'Number of Referrals',
    'Population', 'Satisfaction Score', 'Total Charges', 'Total Extra Data Charges',
    'Total Long Distance Charges', 'Total Refunds', 'Total Revenue'
]].mean().to_dict()

In [18]:
joblib.dump(numeric_defaults, "numeric_defaults.joblib")

['numeric_defaults.joblib']